#Importación de librerias

In [1]:
import os
import json
import random
import pandas as pd
import numpy as np
import cv2
import torch
import torch.nn as nn
import tensorflow as tf
from torch.utils.data import Dataset, DataLoader
from torchvision import models
import albumentations as A
from albumentations.pytorch import ToTensorV2

os.system("pip install -q torch torchvision albumentations opencv-python-headless kaggle pandas")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo de entrenamiento: {DEVICE}")

IMG_SIZE = 260
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

Dispositivo de entrenamiento: cuda


#Descargar de dataset

In [2]:
print("\n1. Verificando dataset...")
if not os.path.exists("./raw_species"):
    os.system("kaggle datasets download -d goelyash/165-different-snakes-species -p ./raw_species --unzip")
DATASET_ROOT = "./raw_species"

csv_train_path = os.path.join(DATASET_ROOT, "Csv", "train.csv")
csv_test_path = os.path.join(DATASET_ROOT, "Csv", "test.csv")

dfs = []
if os.path.exists(csv_train_path): dfs.append(pd.read_csv(csv_train_path))
if os.path.exists(csv_test_path): dfs.append(pd.read_csv(csv_test_path))

df = pd.concat(dfs, ignore_index=True)
df["binomial_clean"] = df["binomial"].astype(str).str.strip()

print("Mapeando ubicación física de imágenes...")
mapa_archivos = {}
for root, _, files in os.walk(DATASET_ROOT):
    for f in files:
        if f.lower().endswith((".jpg", ".jpeg", ".png")):
            path = os.path.join(root, f)
            mapa_archivos[f] = path
            mapa_archivos[os.path.splitext(f)[0]] = path


1. Verificando dataset...
Mapeando ubicación física de imágenes...


#Filtrado de imagenes buscando las mejores para entrenar

In [3]:
print("\n2. Analizando nitidez y calidad visual de las imágenes...")

def calcular_nitidez(ruta_imagen):
    try:
        img = cv2.imread(ruta_imagen, cv2.IMREAD_GRAYSCALE)
        if img is None: return 0.0
        return cv2.Laplacian(img, cv2.CV_64F).var()
    except Exception:
        return 0.0

candidatos_por_especie = {}

for _, row in df.iterrows():
    especie = row["binomial_clean"]
    uuid = str(row.get("UUID", ""))
    path = mapa_archivos.get(uuid) or mapa_archivos.get(f"{uuid}.jpg")

    if path and os.path.exists(path):
        candidatos_por_especie.setdefault(especie, []).append(path)

NUM_ESPECIES_TARGET = 10
FOTOS_POR_ESPECIE = 150

especies_evaluadas = []
for especie, rutas in candidatos_por_especie.items():
    if len(rutas) >= FOTOS_POR_ESPECIE:
        puntajes = [(r, calcular_nitidez(r)) for r in rutas]
        validas = [item for item in puntajes if item[1] > 100.0]

        if len(validas) >= FOTOS_POR_ESPECIE:
            validas.sort(key=lambda x: x[1], reverse=True)
            promedio_calidad = np.mean([x[1] for x in validas[:FOTOS_POR_ESPECIE]])
            especies_evaluadas.append({
                "especie": especie,
                "rutas_filtradas": [x[0] for x in validas[:FOTOS_POR_ESPECIE]],
                "calidad_promedio": promedio_calidad
            })

especies_evaluadas.sort(key=lambda x: x["calidad_promedio"], reverse=True)
top_especies = especies_evaluadas[:NUM_ESPECIES_TARGET]

CLASS_NAMES = sorted([e["especie"] for e in top_especies])
CLASS_TO_IDX = {name: i for i, name in enumerate(CLASS_NAMES)}

items = []
print(f"\n✨ ESPECIES SELECCIONADAS POR MAYOR NITIDEZ Y CALIDAD DE IMAGEN:")
for item in top_especies:
    esp = item["especie"]
    rutas = item["rutas_filtradas"]
    print(f"  - {esp}: {len(rutas)} imágenes de alta calidad (Score promedio: {item['calidad_promedio']:.1f})")
    for r in rutas:
        items.append((r, CLASS_TO_IDX[esp]))


2. Analizando nitidez y calidad visual de las imágenes...

✨ ESPECIES SELECCIONADAS POR MAYOR NITIDEZ Y CALIDAD DE IMAGEN:
  - Lampropeltis calligaster: 150 imágenes de alta calidad (Score promedio: 4146.1)
  - Cemophora coccinea: 150 imágenes de alta calidad (Score promedio: 3418.0)
  - Sistrurus catenatus: 150 imágenes de alta calidad (Score promedio: 3198.2)
  - Sistrurus miliarius: 150 imágenes de alta calidad (Score promedio: 3169.2)
  - Crotalus adamanteus: 150 imágenes de alta calidad (Score promedio: 3152.9)
  - Rhadinaea flavilata: 150 imágenes de alta calidad (Score promedio: 3141.7)
  - Clonophis kirtlandii: 150 imágenes de alta calidad (Score promedio: 2941.1)
  - Heterodon simus: 150 imágenes de alta calidad (Score promedio: 2892.1)
  - Heterodon nasicus: 150 imágenes de alta calidad (Score promedio: 2812.4)
  - Nerodia floridana: 150 imágenes de alta calidad (Score promedio: 2797.1)


#Total de train / test

In [4]:
random.seed(42)
by_class = {}
for path, label in items:
    by_class.setdefault(label, []).append(path)

train_files, val_files = [], []
for label, paths in by_class.items():
    random.shuffle(paths)
    val_count = int(len(paths) * 0.15)
    val_files += [(p, label) for p in paths[:val_count]]
    train_files += [(p, label) for p in paths[val_count:]]

print(f"\nTotal train: {len(train_files)} | Total val: {len(val_files)}")


Total train: 1280 | Total val: 220


#Dataset con Letterbox

In [5]:
def resize_aspect_ratio_pad(image_rgb: np.ndarray, target_size: int = 260) -> np.ndarray:
    h, w = image_rgb.shape[:2]
    scale = target_size / max(h, w)
    new_w, new_h = int(w * scale), int(h * scale)
    interp = cv2.INTER_AREA if scale < 1.0 else cv2.INTER_CUBIC
    resized = cv2.resize(image_rgb, (new_w, new_h), interpolation=interp)
    canvas = np.full((target_size, target_size, 3), 128, dtype=np.uint8)
    top = (target_size - new_h) // 2
    left = (target_size - new_w) // 2
    canvas[top:top + new_h, left:left + new_w] = resized
    return canvas

class SnakeSpeciesDataset(Dataset):
    def __init__(self, items, augment: bool):
        self.items = items
        if augment:
            self.transform = A.Compose([
                A.HorizontalFlip(p=0.5),
                A.RandomBrightnessContrast(p=0.2),
                A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.5, border_mode=cv2.BORDER_CONSTANT),
                A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
                ToTensorV2(),
            ])
        else:
            self.transform = A.Compose([
                A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
                ToTensorV2(),
            ])

    def __len__(self): return len(self.items)

    def __getitem__(self, idx):
        path, label = self.items[idx]
        img_bgr = cv2.imread(path)
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        padded = resize_aspect_ratio_pad(img_rgb, IMG_SIZE)
        out = self.transform(image=padded)
        return out["image"], label

train_loader = DataLoader(SnakeSpeciesDataset(train_files, True), batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(SnakeSpeciesDataset(val_files, False), batch_size=16, shuffle=False, num_workers=2)

/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


#Crear el modelo MobileNetV2

In [6]:
IMG_HEIGHT = IMG_SIZE
IMG_WIDTH = IMG_SIZE
IMG_SIZE_TUPLE = (IMG_HEIGHT, IMG_WIDTH)

base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE_TUPLE + (3,),
    include_top=False,
    weights="imagenet")
base_model.trainable = False

inputs = tf.keras.Input(shape=IMG_SIZE_TUPLE + (3,))
x = base_model(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(len(CLASS_NAMES), activation="softmax")(x)
model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"])

model.summary()

/tmp/ipykernel_1636/2559028239.py:5: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = tf.keras.applications.MobileNetV2(


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 260, 260, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 9, 9, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │        12,810 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,270,794 (8.66 MB)

 Trainable params: 12,810 (50.04 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

#Modelo de alta capacidad con EfficientNet-B2

In [7]:
print("\nCargando EfficientNet-B2...")
model = models.efficientnet_b2(weights=models.EfficientNet_B2_Weights.IMAGENET1K_V1)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(CLASS_NAMES))
model = model.to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
criterion = nn.CrossEntropyLoss(label_smoothing=0.03)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

EPOCHS = 20
best_val_acc = 0.0
os.makedirs("./modelo_especie_out", exist_ok=True)

print("\n🚀 Iniciando entrenamiento optimizado...")
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)

    scheduler.step()
    train_loss = running_loss / len(train_files)

    model.eval()
    correct, total_val = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            out = model(imgs)
            preds = torch.argmax(out, dim=1)
            correct += (preds == labels).sum().item()
            total_val += labels.size(0)

    val_acc = correct / max(total_val, 1)
    print(f"Epoch {epoch+1:02d}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "./modelo_especie_out/modelo_especie.pth")
        print(f"   --> 💾 Guardado modelo con precisión de {val_acc*100:.2f}%")

with open("./modelo_especie_out/class_names.json", "w", encoding="utf-8") as f:
    json.dump(CLASS_NAMES, f, ensure_ascii=False, indent=2)

print(f"\n✨ Proceso terminado. Mejor precisión lograda: {best_val_acc*100:.2f}%")


Cargando EfficientNet-B2...
Downloading: "https://download.pytorch.org/models/efficientnet_b2_rwightman-c35c1473.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b2_rwightman-c35c1473.pth


100%|██████████| 35.2M/35.2M [00:00<00:00, 197MB/s]



🚀 Iniciando entrenamiento optimizado...
Epoch 01/20 | Train Loss: 1.5331 | Val Acc: 0.8000
   --> 💾 Guardado modelo con precisión de 80.00%
Epoch 02/20 | Train Loss: 0.7473 | Val Acc: 0.7864
Epoch 03/20 | Train Loss: 0.5351 | Val Acc: 0.8136
   --> 💾 Guardado modelo con precisión de 81.36%
Epoch 04/20 | Train Loss: 0.4275 | Val Acc: 0.8591
   --> 💾 Guardado modelo con precisión de 85.91%
Epoch 05/20 | Train Loss: 0.3437 | Val Acc: 0.8636
   --> 💾 Guardado modelo con precisión de 86.36%
Epoch 06/20 | Train Loss: 0.3084 | Val Acc: 0.8273
Epoch 07/20 | Train Loss: 0.2664 | Val Acc: 0.8136
Epoch 08/20 | Train Loss: 0.2580 | Val Acc: 0.8091
Epoch 09/20 | Train Loss: 0.2352 | Val Acc: 0.8409
Epoch 10/20 | Train Loss: 0.2340 | Val Acc: 0.8591
Epoch 11/20 | Train Loss: 0.2299 | Val Acc: 0.8500
Epoch 12/20 | Train Loss: 0.2230 | Val Acc: 0.8636
Epoch 13/20 | Train Loss: 0.2199 | Val Acc: 0.8545
Epoch 14/20 | Train Loss: 0.2195 | Val Acc: 0.8273
Epoch 15/20 | Train Loss: 0.2149 | Val Acc: 0.859